In [ ]:
import torch                          # Bibliothèque principale pour le deep learning
import torch.nn as nn                 # Contient les modules de réseaux de neurones
import torch.optim as optim           # Contient les optimiseurs

# =========================
# Générateur (Generator)
# =========================
class Generator(nn.Module):           # Définition du générateur
    def __init__(self, z_dim):        # z_dim = dimension du bruit (latent vector)
        super().__init__()            # Initialise la classe parent

        # Réseau fully connected
        self.model = nn.Sequential(
            nn.Linear(z_dim, 128),    # Transforme le bruit en 128 neurones
            nn.ReLU(),                # Fonction d'activation ReLU

            nn.Linear(128, 256),      # Augmente la dimension
            nn.ReLU(),                # Activation

            nn.Linear(256, 784),      # Sortie (ex: image 28x28 = 784 pixels)
            nn.Tanh()                 # Normalise entre [-1, 1]
        )

    def forward(self, z):             # Passage avant
        return self.model(z)          # Génère une fausse image


# =========================
# Critic (remplace Discriminator)
# =========================
class Critic(nn.Module):
    def __init__(self):
        super().__init__()

        # Réseau similaire mais SANS sigmoid
        self.model = nn.Sequential(
            nn.Linear(784, 256),      # Entrée = image aplatie
            nn.LeakyReLU(0.2),        # Activation (évite neurones morts)

            nn.Linear(256, 128),      
            nn.LeakyReLU(0.2),

            nn.Linear(128, 1)         # Sortie = score réel (pas probabilité)
        )

    def forward(self, x):
        return self.model(x)          # Retourne un score


# =========================
# Hyperparamètres
# =========================
z_dim = 100                          # Dimension du bruit
lr = 0.00005                         # Learning rate (plus petit pour WGAN)
batch_size = 64                      # Taille du batch
clip_value = 0.01                    # Valeur pour le weight clipping


# =========================
# Initialisation des modèles
# =========================
G = Generator(z_dim)                 # Création du générateur
D = Critic()                         # Création du critic


# =========================
# Optimiseurs
# =========================
opt_G = optim.RMSprop(G.parameters(), lr=lr)  # Optimiseur du générateur
opt_D = optim.RMSprop(D.parameters(), lr=lr)  # Optimiseur du critic


# =========================
# Boucle d'entraînement
# =========================
for epoch in range(10):              # Boucle sur les epochs

    for _ in range(100):             # Simulation de batches (à remplacer par DataLoader)

        # Génère des données réelles (ici simulées)
        real = torch.randn(batch_size, 784)

        # =========================
        # Entraînement du Critic
        # =========================
        for _ in range(5):           # On entraîne le critic 5 fois pour 1 fois G

            z = torch.randn(batch_size, z_dim)   # Génère du bruit aléatoire
            fake = G(z)                          # Génère des fausses images

            # Loss WGAN :
            # Maximiser D(real) - D(fake)
            # On minimise l'opposé
            loss_D = -torch.mean(D(real)) + torch.mean(D(fake))

            opt_D.zero_grad()        # Reset des gradients
            loss_D.backward()        # Backpropagation
            opt_D.step()             # Mise à jour des poids

            # =========================
            # Weight Clipping (clé du WGAN)
            # =========================
            for p in D.parameters(): 
                p.data.clamp_(-clip_value, clip_value)  
                # Force les poids à rester dans [-0.01, 0.01]
                # Permet de respecter la contrainte Lipschitz

        # =========================
        # Entraînement du Generator
        # =========================
        z = torch.randn(batch_size, z_dim)   # Nouveau bruit
        fake = G(z)                          # Génération

        # Objectif : maximiser D(fake)
        # Donc minimiser -D(fake)
        loss_G = -torch.mean(D(fake))

        opt_G.zero_grad()        # Reset gradients
        loss_G.backward()        # Backpropagation
        opt_G.step()             # Mise à jour

    # Affichage des pertes
    print(f"Epoch {epoch}, Loss D: {loss_D.item()}, Loss G: {loss_G.item()}")